## مقدمة
في هذا المختبر، ستستخدم لغة SQL لإنشاء جداول وعلاقات في قاعدة بيانات. سنستخدم مكتبة `sqlite3` (المضمّنة مع بايثون) لتنفيذ تعليمات SQL مباشرة. سنبدأ بإنشاء جداول بسيطة ثم نضيف المفاتيح الأساسية والمفاتيح الخارجية والقيود. أخيرًا، ستقوم بتصميم مخطط قاعدة بيانات لمكتبة صغيرة.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# إنشاء اتصال بقاعدة بيانات في الذاكرة (سيتم حذفها بعد إغلاق الدفتر)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
print('تم إنشاء قاعدة البيانات المؤقتة بنجاح.')

## 1. إنشاء الجداول CREATE TABLE
لنبدأ بإنشاء جدول `suppliers` (الموردين) باستخدام جملة CREATE TABLE.
- الصيغة العامة: `CREATE TABLE table_name (column_name data_type, ...);`
- أنواع البيانات الشائعة في SQLite: `INTEGER`, `TEXT`, `REAL`, `BLOB`.
المطلوب: اكتب تعليمة SQL مخزنة في متغير `create_sql` لإنشاء الجدول بالوصف التالي:
- `id` من نوع INTEGER
- `name` من نوع TEXT
- `contact` من نوع TEXT
ثم نفِّذ التعليمة باستخدام cursor.execute.

In [ ]:
# TODO: اكتب تعليمة CREATE TABLE لإنشاء جدول الموردين (suppliers)
create_sql = """
"""
cursor.execute(create_sql)
conn.commit()
print('تم إنشاء الجدول suppliers.')

In [ ]:
# إدراج بعض البيانات لتجربة الجدول
insert_sql = """
INSERT INTO suppliers (id, name, contact) VALUES
(1, 'TechCorp', 'tech@example.com'),
(2, 'PrintWorld', 'print@example.com'),
(3, 'OfficeSupplies', 'office@example.com')
"""
cursor.execute(insert_sql)
conn.commit()
df = pd.read_sql_query('SELECT * FROM suppliers', conn)
df

## 2. المفتاح الأساسي PRIMARY KEY
المفتاح الأساسي (PRIMARY KEY) يضمن فرادة كل صف. في الجدول السابق لم نُعرِّف مفتاحًا أساسيًا.
إنشاء جدول `products` بمفتاح أساسي تلقائي الزيادة (AUTOINCREMENT) وقيود:
- `id`: INTEGER PRIMARY KEY AUTOINCREMENT (مفتاح أساسي تلقائي)
- `name`: TEXT NOT NULL (لا يمكن تركه فارغًا)
- `price`: REAL CHECK (price > 0) (يجب أن يكون السعر أكبر من صفر)
أكمل الكود التالي بكتابة تعليمة CREATE TABLE للجدول products.

In [ ]:
# TODO: أنشئ جدول products بالمفتاح الأساسي AUTOINCREMENT والقيود المذكورة
create_products_sql = """
"""
cursor.execute(create_products_sql)
conn.commit()
print('تم إنشاء الجدول products.')

In [ ]:
# إدراج بيانات منتجات
insert_products_sql = """
INSERT INTO products (name, price) VALUES
('ورق A4', 25.5),
('قلم حبر', 3.0),
('طابعة', 450.0)
"""
cursor.execute(insert_products_sql)
conn.commit()
pd.read_sql_query('SELECT * FROM products', conn)

## 3. المفتاح الخارجي FOREIGN KEY
يُستخدم المفتاح الخارجي لضمان تكامل المراجع بين الجداول. سننشئ جدول `orders` (الطلبات) الذي يرتبط بـ `products` و `suppliers`.
المطلوب: اكتب تعليمة SQL لإنشاء الجدول orders بالحقول:
- order_id INTEGER PRIMARY KEY AUTOINCREMENT
- product_id INTEGER
- supplier_id INTEGER
- quantity INTEGER CHECK (quantity > 0)
- المفاتيح الخارجية: FOREIGN KEY (product_id) REFERENCES products(id), FOREIGN KEY (supplier_id) REFERENCES suppliers(id)
في SQLite، يجب تفعيل المفاتيح الخارجية عبر PRAGMA foreign_keys = ON.

In [ ]:
# تفعيل المفاتيح الخارجية (مطلوب في SQLite)
cursor.execute('PRAGMA foreign_keys = ON')

# TODO: أنشئ جدول orders مع المفاتيح الخارجية المذكورة
create_orders_sql = """
"""
cursor.execute(create_orders_sql)
conn.commit()
print('تم إنشاء الجدول orders.')

In [ ]:
# إدراج طلب (تأكد من صحة المفاتيح)
insert_order_sql = """
INSERT INTO orders (product_id, supplier_id, quantity) VALUES
(1, 1, 10),
(2, 2, 50)
"""
cursor.execute(insert_order_sql)
conn.commit()
pd.read_sql_query('SELECT * FROM orders', conn)

## 4. القيود: UNIQUE, NOT NULL, CHECK
الآن سنستخدم قيودًا إضافية: UNIQUE لمنع تكرار القيم، وNOT NULL لمنع القيم الفارغة، وCHECK لتطبيق شرط منطقي.
المطلوب: أنشئ جدول `library_members` (أعضاء المكتبة) بالحقول:
- member_id INTEGER PRIMARY KEY AUTOINCREMENT
- email TEXT NOT NULL UNIQUE
- join_date TEXT DEFAULT CURRENT_DATE
- CONSTRAINT email_length_check CHECK (length(email) > 5)
اكتب تعليمة SQL في المتغير member_sql.

In [ ]:
# TODO: اكتب تعليمة CREATE TABLE لجدول library_members تتضمن القيود المذكورة
member_sql = """
"""
cursor.execute(member_sql)
conn.commit()
print('تم إنشاء جدول أعضاء المكتبة.')

In [ ]:
# اختبار القيد: إدراج عضو بنبريد إلكتروني صالح
try:
    cursor.execute("INSERT INTO library_members (email) VALUES ('user@library.com')")
    conn.commit()
    print('تم إدراج العضو الأول بنجاح.')
except sqlite3.IntegrityError as e:
    print('فشل الإدراج:', e)

# TODO: حاول إدراج عضو ببريد إلكتروني مكرر لترى خطأ القيد
try:
    cursor.execute("INSERT INTO library_members (email) VALUES ('')")
    conn.commit()
    print('تم إدراج العضو الثاني بنجاح.')
except sqlite3.IntegrityError as e:
    print('فشل الإدراج المتوقع:', e)

## 5. مشروع: صمم قاعدة بيانات لمكتبة
كمشروع نهائي لهذه الوحدة، قم بإنشاء مخطط قاعدة بيانات بسيط لمكتبة يحتوي على جداول `authors`, `books`, و `borrowers` مع المفاتيح والقيود المناسبة.
المتطلبات:
- جدول `authors`: `id` (INTEGER PRIMARY KEY AUTOINCREMENT), `name` (TEXT NOT NULL).
- جدول `books`: `id` (INTEGER PRIMARY KEY AUTOINCREMENT), `title` (TEXT NOT NULL), `author_id` (INTEGER), `isbn` (TEXT UNIQUE), `FOREIGN KEY (author_id) REFERENCES authors(id)`.
- جدول `borrowers`: `id` (INTEGER PRIMARY KEY AUTOINCREMENT), `name` (TEXT NOT NULL), `email` (TEXT NOT NULL UNIQUE).
اكتب التعليمات اللازمة لإنشاء هذه الجداول في الخلية التالية.

In [ ]:
# TODO: اكتب تعليمات CREATE TABLE للجداول authors, books, borrowers بالترتيب
create_authors = """
"""
cursor.execute(create_authors)

create_books = """
"""
cursor.execute(create_books)

create_borrowers = """
"""
cursor.execute(create_borrowers)

conn.commit()
print('تم إنشاء جداول المكتبة بنجاح!')

### اختبار المخطط: أدرج بعض البيانات وجرب استعلامًا
بعد إنشاء الجداول، أدخل بعض الكتب والمؤلفين والمستعيرين ثم اعرض النتائج.

In [ ]:
# إدراج بيانات
cursor.execute("INSERT INTO authors (name) VALUES ('نجيب محفوظ'), ('أحمد خالد توفيق')")
cursor.execute("INSERT INTO books (title, author_id, isbn) VALUES ('أولاد حارتنا', 1, 'ISBN-001'), ('يوتوبيا', 2, 'ISBN-002')")
cursor.execute("INSERT INTO borrowers (name, email) VALUES ('علي أحمد', 'ali@mail.com'), ('منى محمد', 'mona@mail.com')")
conn.commit()

# استعلام يعرض الكتب مع أسماء مؤلفيها
query = '''
SELECT b.title, a.name AS author, b.isbn
FROM books b
JOIN authors a ON b.author_id = a.id
'''
pd.read_sql_query(query, conn)